In [ ]:
"""SS26 Kaggle judge — standalone, notebook-safe.

Attach dataset ``maze-maps-csess26`` rồi chạy cell này. Thí sinh chỉ nộp CSV:
id,q_forward,q_rotate_left,q_rotate_right
"""

import json
import math
from pathlib import Path

import pandas as pd
import pandas.api.types


class ParticipantVisibleError(Exception):
    """Lỗi submission được phép hiển thị cho thí sinh."""


N_CP_MAX = 3
N_ROWS = 5184
MAX_STEPS_INFER = 200
POLICY_COLUMNS = ("q_forward", "q_rotate_left", "q_rotate_right")
ACTIONS = ("forward", "rotate left", "rotate right")
HEADING_IDX = {"N": 0, "W": 1, "E": 2, "S": 3}
TURN_LEFT = {"N": "W", "W": "S", "S": "E", "E": "N"}
TURN_RIGHT = {"N": "E", "E": "S", "S": "W", "W": "N"}
DIR_DELTA = {"N": (0, 1), "S": (0, -1), "E": (1, 0), "W": (-1, 0)}
OPPOSITE = {"N": "S", "S": "N", "E": "W", "W": "E"}

KAGGLE_INFER_MAP_DIRS = (
    Path("/kaggle/input/datasets/namphongnguynhu/maze-maps-csess26/map/infer"),
    Path("/kaggle/input/maze-maps-csess26/map/infer"),
)
LOCAL_INFER_MAP_DIR = Path("map/infer")


def _list_infer_map_files():
    for directory in KAGGLE_INFER_MAP_DIRS + (LOCAL_INFER_MAP_DIR,):
        if directory.is_dir():
            files = sorted(directory.glob("*.json"))
            if files:
                return files
    raise RuntimeError(
        "Không tìm thấy map inference. Hãy attach dataset maze-maps-csess26."
    )


def _load_map(path):
    with path.open("r", encoding="utf-8") as file:
        spec = json.load(file)

    width = int(spec["width"])
    height = int(spec["height"])
    start = tuple(int(v) for v in spec["start"])
    goal = tuple(int(v) for v in spec["goal"])
    checkpoints = [tuple(int(v) for v in cp) for cp in spec.get("checkpoints", [])]
    walls = set()
    for wall in spec.get("walls", []):
        if isinstance(wall, dict):
            x, y, direction = int(wall["x"]), int(wall["y"]), wall["dir"]
        else:
            x, y, direction = int(wall[0]), int(wall[1]), wall[2]
        walls.add((x, y, direction))
        nx, ny = _neighbor(x, y, direction)
        if _is_valid(nx, ny, width, height):
            walls.add((nx, ny, OPPOSITE[direction]))

    return {
        "name": spec.get("name", path.stem),
        "width": width,
        "height": height,
        "start": start,
        "goal": goal,
        "checkpoints": checkpoints,
        "walls": walls,
    }


def _neighbor(x, y, direction):
    dx, dy = DIR_DELTA[direction]
    return x + dx, y + dy


def _is_valid(x, y, width, height):
    return 0 <= x < width and 0 <= y < height


def _is_blocked(sim_map, x, y, direction):
    nx, ny = _neighbor(x, y, direction)
    if not _is_valid(nx, ny, sim_map["width"], sim_map["height"]):
        return True
    return (x, y, direction) in sim_map["walls"]


def _manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def _dist_trend(previous, current):
    if current < previous:
        return 1
    if current > previous:
        return -1
    return 0


def _make_robot(sim_map):
    sx, sy = sim_map["start"]
    return {
        "x": sx,
        "y": sy,
        "direction": "N",
        "obstacles": {},
        "prev_goal": None,
        "prev_cp": [None] * N_CP_MAX,
        "goal_trend": 0,
        "cp_trends": [0] * N_CP_MAX,
        "has_prev_node": False,
        "cp_visited": [False] * len(sim_map["checkpoints"]),
    }


def _remember_facing_edge(robot, sim_map):
    x, y, direction = robot["x"], robot["y"], robot["direction"]
    blocked = 1 if _is_blocked(sim_map, x, y, direction) else 0
    robot["obstacles"][(x, y, direction)] = blocked
    nx, ny = _neighbor(x, y, direction)
    if _is_valid(nx, ny, sim_map["width"], sim_map["height"]):
        robot["obstacles"][(nx, ny, OPPOSITE[direction])] = blocked


def _obstacle_bits(robot):
    x, y = robot["x"], robot["y"]
    memory = robot["obstacles"]
    n = memory.get((x, y, "N"), 0)
    w = memory.get((x, y, "W"), 0)
    e = memory.get((x, y, "E"), 0)
    s = memory.get((x, y, "S"), 0)
    return n * 8 + w * 4 + e * 2 + s


def _encode_state(robot):
    trends = [robot["goal_trend"]] + robot["cp_trends"][:N_CP_MAX]
    packed = sum((trend + 1) * (3 ** index) for index, trend in enumerate(trends))
    return _obstacle_bits(robot) * (3 ** 4) * 4 + packed * 4 + HEADING_IDX[robot["direction"]]


def _get_policy(encoded_state, q_table):
    row = q_table[encoded_state]
    best = 0
    for index in range(1, len(row)):
        if row[index] > row[best]:
            best = index
    return ACTIONS[best]


def _clear_trends(robot):
    robot["goal_trend"] = 0
    robot["cp_trends"] = [0] * N_CP_MAX


def _execute_action(robot, sim_map, action_name):
    if action_name == "rotate left":
        robot["direction"] = TURN_LEFT[robot["direction"]]
        _clear_trends(robot)
        _remember_facing_edge(robot, sim_map)
        return False

    if action_name == "rotate right":
        robot["direction"] = TURN_RIGHT[robot["direction"]]
        _clear_trends(robot)
        _remember_facing_edge(robot, sim_map)
        return False

    x, y = robot["x"], robot["y"]
    direction = robot["direction"]
    if _is_blocked(sim_map, x, y, direction):
        _remember_facing_edge(robot, sim_map)
        _clear_trends(robot)
        return True

    current = (x, y)
    previous_goal = _manhattan(current, sim_map["goal"])
    previous_cps = [_manhattan(current, cp) for cp in sim_map["checkpoints"]]
    nx, ny = _neighbor(x, y, direction)
    robot["x"], robot["y"] = nx, ny

    if robot["has_prev_node"]:
        robot["goal_trend"] = _dist_trend(previous_goal, _manhattan((nx, ny), sim_map["goal"]))
        cp_trends = [0] * N_CP_MAX
        for index in range(min(N_CP_MAX, len(sim_map["checkpoints"]))):
            if not robot["cp_visited"][index]:
                current_distance = _manhattan((nx, ny), sim_map["checkpoints"][index])
                cp_trends[index] = _dist_trend(previous_cps[index], current_distance)
        robot["cp_trends"] = cp_trends
    else:
        _clear_trends(robot)

    for index, checkpoint in enumerate(sim_map["checkpoints"]):
        if (nx, ny) == checkpoint:
            robot["cp_visited"][index] = True
    robot["has_prev_node"] = True
    _remember_facing_edge(robot, sim_map)
    return False


def _submission_to_q_table(submission, row_id_column_name):
    if row_id_column_name not in submission.columns:
        raise ParticipantVisibleError("Submission thiếu cột ID '%s'." % row_id_column_name)

    policy = submission.drop(columns=[row_id_column_name])
    if tuple(policy.columns) != POLICY_COLUMNS:
        raise ParticipantVisibleError(
            "Các cột policy phải đúng thứ tự: %s." % ", ".join(POLICY_COLUMNS)
        )
    if len(policy) != N_ROWS:
        raise ParticipantVisibleError(
            "Policy phải có đúng %d dòng, hiện có %d." % (N_ROWS, len(policy))
        )
    for column in POLICY_COLUMNS:
        if not pandas.api.types.is_numeric_dtype(policy[column]):
            raise ParticipantVisibleError("Cột '%s' phải chứa số." % column)

    q_table = policy.astype(float).values.tolist()
    if any(not math.isfinite(value) for row in q_table for value in row):
        raise ParticipantVisibleError("Policy không được chứa NaN hoặc giá trị vô hạn.")
    return q_table


def calculate_map_score(sim_map, q_table):
    robot = _make_robot(sim_map)
    _remember_facing_edge(robot, sim_map)

    visited_checkpoints = set()
    if sim_map["start"] in sim_map["checkpoints"]:
        visited_checkpoints.add(sim_map["start"])

    collision = False
    goal_reached = robot["x"] == sim_map["goal"][0] and robot["y"] == sim_map["goal"][1]
    step_count = 0

    for step in range(1, MAX_STEPS_INFER + 1):
        if goal_reached:
            break
        action_name = _get_policy(_encode_state(robot), q_table)
        collision = _execute_action(robot, sim_map, action_name)
        step_count = step

        position = (robot["x"], robot["y"])
        if position in sim_map["checkpoints"]:
            visited_checkpoints.add(position)
        goal_reached = position == sim_map["goal"]
        if collision or goal_reached:
            break

    return float(
        (400 if goal_reached else 0)
        + 100 * len(visited_checkpoints)
        - (100 if collision else 0)
        - 2 * step_count
    )


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """Chạy policy trên toàn bộ map inference và trả tổng điểm (higher is better)."""
    del solution
    q_table = _submission_to_q_table(submission, row_id_column_name)
    total = sum(calculate_map_score(_load_map(path), q_table) for path in _list_infer_map_files())
    if not math.isfinite(total):
        raise RuntimeError("Điểm tính được không hợp lệ.")
    return float(total)
